Mesh independence fig is in:
 - WorkPackages/TMCJ-Contact/notebooks/feaAnalysis/meshInd/study1-50000R/study1-box.ipynb 
   
Results comparison fig is in:
 - WorkPackages/TMCJ-Contact/notebooks/feaAnalysis/accuracyCheck/14548Rneutral/14548RneutralStudy3-again.ipynb

# Cartilage generation fig

In [ ]:
import numpy as np
import pandas as pd
import pyvista as pv
import gdist
from scipy.spatial.distance import cdist
from pathlib import Path
import subprocess

from phd_helpers.CartilageGeneration import (
    mesh_checks, bone_cartilage_checks, get_outward_normal_mask, flip_faces, taper_f,
    interp_vecs, get_triangle_adjacency, flood_fill_cells, remove_normals, fill_holes_pmf
)
from phd_helpers.paths import find_corresponding_cells, identical_points_count, get_boundary

In [10]:
import subprocess
path_MeshPipeline_main = '../../../MeshPipeline/main.py'
subprocess.run(["python", path_MeshPipeline_main])


Updating parameters.json
	Wrote /Users/maro/Library/CloudStorage/OneDrive-UniversityofLeeds/PhD-wrist/WorkPackages/WorkPackages/TMCJ-Contact/Computational/MeshPipeline/set_parameters/parameters.json


Full parameter file saved to outputs/smoothRemesh/params/full_params.json

SUBJECT: 50017L
	BONES: tpm-mc1
		STEP: 2Dmesh
			RUN ID: -0
			Runtime: 6.328s - ok
	BONES: mc1-tpm
		STEP: 2Dmesh
			RUN ID: -0
			Runtime: 2.204s - ok


CompletedProcess(args=['python', '../../../MeshPipeline/main.py'], returncode=0)

In [15]:
from phd_helpers.paths import get_subject_stl_path, get_mesh, PROJECT_ROOT
from phd_helpers.CartilageGeneration import get_min_df_fast

subject, sideL = '50017', 'L'
bone = 'tpm'
ar_bone = 'mc1'
poses = ['adduction','abduction','flexion','extension','pinch','grasp','jar','neutral']
max_gap_cartilage = 2

mesh_path = Path(f'outputs/smoothRemesh/meshes/{subject+sideL}')
bone_mesh = pv.read(mesh_path / ('tpm-mc1/2Dmesh/bone_remesh-0.obj'))
arbone_mesh = pv.read(mesh_path / ('mc1-tpm/2Dmesh/bone_remesh-0.obj'))

stl_path = get_subject_stl_path(subject, sideL)
min_df_orig = get_min_df_fast(stl_path, bone, ar_bone, bone_mesh, arbone_mesh, poses, max_gap_cartilage)

In [ ]:
remesh_cartilage = True
cartilage_remesh_name = f'CartilageCap.obj' # cartilage cap mesh file name
cgal_path = PROJECT_ROOT / 'WorkPackages/TMCJ-Contact/Computational/MeshPipeline/cpp/2Dmesh'
cgal_input_path = cgal_path / f'inputs/fb_input/{cartilage_remesh_name}' # input path for CGAL C++ remeshing

taper_width = 1.5 # width of cartilage taper region 
#max_height, # max height of cartilage in taper region
p_h = 8.5 # shape of taper height (1 = linear , higher = steeper taper)
p_v = 1 # shape of vector ratio (1 = linear)
cartilage_smooth_iters = 100 # need to look at this, currently uses laplacian  ##################### check this #############
edge_length = 0.2 # target edge length of cartilage remesh
n_iters = 5 # n isotropic remeshing iterations for cartilage remesh
clamp_height = 0.015 # minimum cartilage height

min_df = min_df_orig.copy()

################# MESH STUFF #################
bone_mesh['Normals'] = bone_mesh.compute_normals(point_normals=True, cell_normals=False)['Normals']
bone_mesh['bone_id'] = np.arange(bone_mesh.n_points)
bone_mesh['bone_cell_id'] = np.arange(bone_mesh.n_cells)
################# MESH STUFF #################


################# COMPUTE TAPER REGION #################
# extract mesh of cartilage points on bone mesh (makes gdist computation faster) - (bone-cartilage interface mesh)
inter_mesh1 = bone_mesh.extract_points(min_df['bone_id'], adjacent_cells=False).extract_surface(algorithm=None)

# JUST REMOVE ANY ISLANDS
conn = inter_mesh1.connectivity(label_regions=True)
region_ids = conn.cell_data["RegionId"]
island_ids, counts = np.unique(region_ids, return_counts=True)
largest_region_id = island_ids[np.argmax(counts)]
inter_mesh = inter_mesh1.extract_cells(np.where(region_ids == largest_region_id)[0]).extract_surface(algorithm=None)
inter_mesh['inter_cell_ids'] = np.arange(inter_mesh.n_cells)

# remove any missing points due to extract geometry (incomplete triangles) and islands
missing_mask = ~np.isin(min_df['bone_id'], inter_mesh['bone_id'])
min_df.drop(min_df['bone_id'][missing_mask].index.values, inplace=True)

# useful values from min_df (min_df does not change after this point - final change was at min_df.drop(missing))
midpoint_dist = min_df['dist'] / 2 # distance of midpoint between two bones for each pair of closest points
midpoints = np.array(min_df['midpoint'].tolist()) # midpoint coordinates
#ar_points = np.array(min_df['ar_point'].tolist()) # closest points on ar bone to each bone point


# get boundary of cartilage on inter mesh
inter_boundary = get_boundary(inter_mesh)
boundary_mask_inter = np.isin(inter_mesh['bone_id'], inter_boundary['bone_id']) 
boundary_ids = np.arange(inter_mesh.n_points)[boundary_mask_inter] # on inter_mesh

# minimum geo dist of every node from closest source_idx
geo_dists = gdist.compute_gdist(
    inter_mesh.points.astype(np.float64),
    inter_mesh.faces.reshape(-1, 4)[:, 1:].astype(np.int32),
    source_indices=boundary_ids.astype(np.int32), 
) 

# get mask of nodes within taper width and below taper function
#taper_heights = taper_f(geo_dists, taper_width, max_height, p=p_h)
#taper_mask = (taper_heights <= min_df['dist'] / 2) & (geo_dists<=taper_width)
taper_mask = geo_dists<=taper_width # new

# get taper points mesh (makes computation much faster) - mesh of only taper region
taper_mask_inter = np.isin(inter_mesh['bone_id'], min_df['bone_id'][taper_mask])
taper_mesh = inter_mesh.extract_points(taper_mask_inter, adjacent_cells=False).extract_surface(algorithm=None)
    # assign non-taper cells that lie on "pinched" islands to taper region
not_taper_mesh = inter_mesh.extract_cells(taper_mesh['inter_cell_ids'], invert=True).extract_surface(algorithm=None)
_, adjacency = get_triangle_adjacency(not_taper_mesh)
start_face = not_taper_mesh.find_closest_cell(np.mean(not_taper_mesh.points, axis=0)) # not best way of doing this!
inner_cells = flood_fill_cells(not_taper_mesh, start_face, get_boundary(not_taper_mesh).lines.reshape(-1, 3)[:, 1:], adjacency)
    # final taper mesh #
taper_mesh = inter_mesh.extract_cells(not_taper_mesh['inter_cell_ids'][inner_cells], invert=True).extract_surface(algorithm=None)
taper_mesh['taper_point_id'] = np.arange(taper_mesh.n_points)
    # remove any missing points from taper_mask after extracting geometry and islands - not ideal but quicker than using inter mesh
taper_mask = np.isin(min_df['bone_id'], taper_mesh['bone_id'])
#taper_geo_dists = geo_dists[taper_mask]

# get taper mesh boundaries
taper_boundary = get_boundary(taper_mesh)
boundary_outer_mask_tb = np.isin(taper_boundary['bone_id'], inter_boundary['bone_id']) # on taper_boundary

# get taper boundary innner and outer nodes on taper mesh
taper_outer_ids = np.sort(taper_boundary['taper_point_id'][boundary_outer_mask_tb]) # on taper mesh
taper_inner_ids = np.sort(taper_boundary['taper_point_id'][~boundary_outer_mask_tb]) # on taper mesh

# ids and distances of boundary nodes that are closest to each taper node
d_taper_outer = cdist(taper_mesh.points, taper_mesh.points[taper_outer_ids])
near_taper_outer_D = d_taper_outer.min(axis=1)
#near_taper_outer_ids = taper_outer_ids[np.argmin(d_taper_outer, axis=1)]

d_taper_inner = cdist(taper_mesh.points, taper_mesh.points[taper_inner_ids])
near_taper_inner_D = d_taper_inner.min(axis=1)
near_taper_inner_ids = taper_inner_ids[np.argmin(d_taper_inner, axis=1)]

# Distance fraction of each taper_mesh point from closest outer_node to closest_inner node
taper_Df = (near_taper_outer_D) / (near_taper_outer_D + near_taper_inner_D)

# get nearest inner node midpoint heights
taper_inner_mask_c = np.isin(min_df['bone_id'], taper_mesh['bone_id'][taper_inner_ids]) # on c_mesh
inner_node_midpoint_heights = midpoint_dist.values[taper_inner_mask_c]
near_inner_node_midpoint_heights = inner_node_midpoint_heights[np.searchsorted(taper_inner_ids, near_taper_inner_ids)]

# set taper node heights - should be based on interpolated height really - but as long as mesh is dense this is fine...
taper_heights = taper_f(taper_Df, 1, near_inner_node_midpoint_heights, p=p_h) # non-linear

# set vector directions
midpoint_vecs = midpoints - bone_mesh.points[min_df['bone_id']] # vector from bone point to midpoint
taper_vecs_mid = midpoint_vecs[taper_mask] # midpoint_vecs
taper_vecs_norm = bone_mesh['Normals'][min_df['bone_id']][taper_mask] # normal vecs
vec_dirs = taper_f(taper_Df, 1, 1, p=p_v)
taper_vecs = interp_vecs(taper_vecs_norm, taper_vecs_mid, vec_dirs)

# get taper coords
taper_points = bone_mesh.points[min_df['bone_id']][taper_mask]
taper_points = taper_points + taper_heights.reshape(-1, 1)*taper_vecs

# get taper points
points_tapered = midpoints.copy()
points_tapered[taper_mask] = taper_points

#closest_points, distances, cell_ids = get_trimesh(bone_mesh).nearest.on_surface(points_tapered)
################# COMPUTE TAPER REGION #################

################# MESH TAPER REGION #################
# map faces from bone mesh to extruded taper region points
# mesh curves down in taper region and delaunay doesn't like that
tapered_mesh = pv.PolyData(taper_points, taper_mesh.faces)
################# MESH TAPER REGION #################

################# MESH INNER REGION #################
tapered_edge = get_boundary(tapered_mesh)

inner_edge = tapered_edge.extract_points(~boundary_outer_mask_tb).extract_surface(algorithm=None)
#inner_points = pv.PolyData(midpoints[~taper_mask])
inner_mesh = pv.PolyData(np.vstack( (inner_edge.points, midpoints[~taper_mask]) ), lines=inner_edge.lines)

# alpha for coarse meshes wants to be slightly lower and vice versa - I think
inner_mesh = inner_mesh.delaunay_2d(edge_source=inner_edge, alpha=0.9).triangulate()
inner_mesh = inner_mesh.fill_holes(inner_mesh.area/20)
#inner_mesh.lines = inner_edge.lines # reset edge lines to remove delaunay leftover lines
inner_mesh.lines = np.empty(0, dtype='int64') # remove all lines for now cos they show up in mesh.faces

# remove cells that lie outside of the inner region boundary
edge_map, adjacency = get_triangle_adjacency(inner_mesh)
#start_face = inner_mesh.find_closest_cell(np.mean(inner_mesh.points, axis=0))
start_face = inner_mesh.find_closest_cell(inner_mesh.center) # new 28/03/26
inner_cells = flood_fill_cells(inner_mesh, start_face, inner_edge.lines.reshape(-1, 3)[:, 1:], adjacency)
inner_mesh_clean = pv.PolyData(inner_mesh.points, inner_mesh.faces.reshape(-1, 4)[inner_cells])
# should maybe be calling remove unused points here, lines bring their own points that are left behind?
################# MESH INNER REGION #################

################# COMBINE INNER MESH AND TAPER MESH #################
#edge_check1 = np.isin(inner_edge.points, tapered_mesh.points).all()
edge_check1 = identical_points_count(inner_edge.points, tapered_mesh.points) == inner_edge.n_points #*** new
#edge_check2 = np.isin(inner_edge.points, inner_mesh_clean.points).all()
edge_check2 = identical_points_count(inner_edge.points, inner_mesh_clean.points) == inner_edge.n_points #*** new

inner_mesh_clean['inner_cells'] = np.full(inner_mesh_clean.n_cells, 1)
tapered_mesh['inner_cells'] = np.full(tapered_mesh.n_cells, 0)
mesh = inner_mesh_clean + tapered_mesh # full cartilage cap mesh
    # check if there are any holes at boundary between inner mesh and tapered mesh and try to fill them
if get_boundary(mesh).connectivity()['RegionId'].any():
    mesh = fill_holes_pmf(mesh, nbe=20)
    mesh['inner_cells'] = np.ones(mesh.n_cells, dtype=int)
    mesh['inner_cells'][find_corresponding_cells(mesh, tapered_mesh, raise_error=True)] = 0
else:
    if not mesh.n_points == midpoints.shape[0]: # had to move to else cos pymeshfix messes with stuff sometimes
        raise AssertionError('Not all midpoints in mesh (& no dupes)')

if not edge_check1:
    raise AssertionError('Not all boundary points in tapered mesh')
if not edge_check2:
    raise AssertionError('Not all boundary points in inner mesh')

# check for flat faces
#closest_points, implicit_distances, cell_ids = get_trimesh(bone_mesh).nearest.on_surface(mesh.points)
_, ps = bone_mesh.find_closest_cell(mesh.points, return_closest_point=True)
implicit_distances = np.linalg.norm(mesh.points - ps, axis=1) #*** changed from line above

mesh_faces = mesh.faces.reshape(-1, 4)[:, 1:]
flat_face_mask = (implicit_distances[mesh_faces] <= 1e-12).all(axis=1) #*** changed from ==0 to <=1e-12
#flat_face_ids = np.where(flat_face_mask)[0]

# remove flat faces and leftover points - occur when all 3 vertices lie on the boundary
mesh_clean = pv.PolyData(mesh.points, mesh.faces.reshape(-1, 4)[~flat_face_mask]).compute_normals(auto_orient_normals=True)
mesh_clean.remove_unused_points(inplace=True)
mesh_clean['mesh_clean_id'] = np.arange(mesh_clean.n_points)
# get array of inner points/cells on mesh_clean 
mesh_clean['inner_cells'] = mesh['inner_cells'][find_corresponding_cells(mesh, mesh_clean)]
mesh_clean['inner_points'] = np.full(mesh_clean.n_points, 1)
mesh_clean['inner_points'][np.unique(mesh_clean.faces.reshape(-1, 4)[:, 1:][np.where(mesh_clean['inner_cells']==0)[0]])] = 0
#mesh_clean['inner_points'] = np.zeros(mesh_clean.n_points, dtype=int)
#for p in inner_mesh_clean.points:
#    mesh_clean['inner_points'][mesh_clean.find_closest_point(p)] = 1


# Remove flat faces from interface mesh (final interface mesh)
if flat_face_mask.any():
    interface_mesh = inter_mesh.extract_cells(find_corresponding_cells(inter_mesh, mesh.extract_cells(flat_face_mask)), invert=True)
else:
    interface_mesh = inter_mesh.copy(deep=True)


# get edge points on mesh and bone_mesh
mesh_clean_edge = get_boundary(mesh_clean)
mesh_clean_edge_mask = np.isin(mesh_clean['mesh_clean_id'], mesh_clean_edge['mesh_clean_id']) # on mesh_clean
mesh_edge_ids = mesh_clean['mesh_clean_id'][mesh_clean_edge_mask] # on mesh_clean
mesh_edge_points = mesh_clean.points[mesh_clean_edge_mask] # on mesh_clean

#print('Checking mesh boundary has not changed')
#bone_mesh_edge_ids = check_points_still_there(bone_mesh, mesh_edge_points)
_, bone_mesh_edge_ids, _ = identical_points_count(bone_mesh.points, mesh_edge_points, return_indices=True) #*** new
bone_mesh_edge_ids = np.sort(bone_mesh_edge_ids)
if len(bone_mesh_edge_ids) != len(mesh_edge_points):
    raise AssertionError("Mesh boundary has changed")


if (mesh.n_cells - mesh_clean.n_cells) != flat_face_mask.sum():
    raise AssertionError("Not all flat faces removed")
################# COMBINE INNER MESH AND TAPER MESH #################
print('Created cartilage cap')

print('Smoothing cartilage cap')
################# SMOOTH CARTILAGE MESH CAP #################
cartilage_cap = mesh_clean.smooth(
n_iter=cartilage_smooth_iters,
feature_angle=180, # prevent anything from being feature cos I don't understant feature_smoothing arg...
boundary_smoothing = False, # keeps boundary fixed-ish
feature_smoothing = False # prevents feature edges from being identified - so they get smoothed?... (idk stupid)
)
cartilage_cap_edge = get_boundary(cartilage_cap)

if cartilage_cap_edge.n_points != mesh_clean_edge.n_points:
    raise AssertionError(f'{cartilage_cap_edge.n_points - mesh_clean_edge.n_points} Boundary points lost during smoothing')
#_ = check_points_still_there(mesh_clean_edge, cartilage_cap_edge.points)

# put edge points back so they are numerically identical
cartilage_cap.points[mesh_edge_ids] = bone_mesh.points[bone_mesh_edge_ids]

################# SMOOTH CARTILAGE MESH CAP #################

################# CLAMP CARTILAGE HEIGHT #################
if clamp_height:
    def clamp_cartilage(cartilage_mesh, bone_mesh, min_height):
        mesh = cartilage_mesh.copy(deep=True)

        inner = mesh.extract_cells(mesh['inner_cells']==1).extract_surface(algorithm=None)
        _, close_points = bone_mesh.find_closest_cell(inner.points, return_closest_point=True)
        vecs = inner.points - close_points
        ds = np.linalg.norm(vecs, axis=1)
        dirs = vecs / np.linalg.norm(vecs, axis=1, keepdims=True)
        clamp_mask = ds < min_height
        if clamp_mask.any():
            print('Clamping')
            inner.points[clamp_mask] = close_points[clamp_mask] + dirs[clamp_mask] * min_height
            mesh.points[inner['mesh_clean_id']] = inner.points
        return mesh

    mesh_clean = clamp_cartilage(mesh_clean, bone_mesh, clamp_height) # clamp mesh clean aswel for returning and saving
    cartilage_cap = clamp_cartilage(cartilage_cap, bone_mesh, clamp_height)
################# CLAMP CARTILAGE HEIGHT #################

if remesh_cartilage:
    ################# REMESH CARTILAGE #################
    # remesh cartilage with CGAL fixed boundary - check boundary hasn't moved (and put points back anyway)
    # this is the final step in creating the cartilage cap mesh (just need to attach to bone after this - trivial)

    print('Writing mesh to remeshing input directory')
    cartilage_cap.save(cgal_input_path)

    print('Remeshing cartilage cap')

    cgal_path = cgal_input_path.parent.parent.parent
    cgal_output_path = cgal_path / 'outputs/fb_output' / cgal_input_path.name
    args = [
        str(cgal_path / "bin/fixed_boundary"),
        str(cgal_input_path),  # path to input mesh
        str(cgal_output_path), # path to output mesh
        str(edge_length), # target edge length
        str(n_iters), # number of CGAL isotropic remeshing iterations
        ]

    result = subprocess.run(args, text=True)
    result.check_returncode()  # raise after printing




    # load remeshed cartilage cap
    cartilage_remesh = pv.read(cgal_output_path)
    cartilage_remesh['remesh_ids'] = np.arange(cartilage_remesh.n_points)

    # check boudnary hasn't moved
    #print('Checking mesh boundary has not changed')
    cartilage_remesh_edge = get_boundary(cartilage_remesh)
    if cartilage_remesh_edge.n_points != cartilage_cap_edge.n_points:
        raise ValueError(f'{cartilage_remesh_edge.n_points - cartilage_cap_edge.n_points} Boundary points lost in remeshing')
    
    #cartilage_cap_edge_ids = check_points_still_there(cartilage_cap_edge, cartilage_remesh_edge.points)
    remesh_edge_dists = cdist(cartilage_cap_edge.points, cartilage_remesh_edge.points) #*** new
    cartilage_cap_edge_ids = np.argmin(remesh_edge_dists, axis=0)
    closest_dists = np.min(remesh_edge_dists, axis=0)
    if (closest_dists > 1e-5).any():
        raise AssertionError(f"Cartilage boundary points moved during remeshing: {max(closest_dists):.5f} mm")

    # put edge points back so they are numerically identical
    cartilage_remesh.points[cartilage_remesh_edge['remesh_ids']] = cartilage_cap_edge.points[cartilage_cap_edge_ids]

    # measure cartilage height and store in mesh
    cartilage_remesh_height = cartilage_remesh.compute_implicit_distance(bone_mesh)['implicit_distance']
    if (cartilage_remesh_height < 0).any():
        raise AssertionError('Not all cartilage points above bone surface') # detect interference

    # add cartilge taper/inner region array
    cartilage_remesh['inner_cells'] = mesh_clean['inner_cells'][mesh_clean.find_closest_cell(cartilage_remesh.cell_centers().points)]
    
    ################# REMESH CARTILAGE #################
else:
    cartilage_remesh = cartilage_cap.copy(deep=True)
    cartilage_remesh['remesh_ids'] = np.arange(cartilage_remesh.n_points)
    cartilage_remesh_edge = get_boundary(cartilage_remesh)

print('\nAttaching cartilage cap')


################# GET FINAL COMBINED MESH WITH REGION IDs #################

# assign scalar data - region_id
bone_mesh['region_id'] = np.ones(bone_mesh.n_cells, dtype=int)
bone_mesh['region_id'][find_corresponding_cells(bone_mesh, interface_mesh, raise_error=True)] = 3
cartilage_remesh['region_id'] = np.full(cartilage_remesh.n_cells, 2)

# create combined mesh
bone_mesh['inner_cells'] = np.full(bone_mesh.n_cells, -1)
combined_mesh = bone_mesh + cartilage_remesh
combined_mesh.compute_normals(inplace=True, consistent_normals=False)

combined_mesh['inner_points'] = np.full(combined_mesh.n_points, -1)
combined_mesh['inner_points'][np.unique(combined_mesh.faces.reshape(-1, 4)[:, 1:][np.where(combined_mesh['inner_cells']==1)[0]])] = 1
combined_mesh['inner_points'][np.unique(combined_mesh.faces.reshape(-1, 4)[:, 1:][np.where(combined_mesh['inner_cells']==0)[0]])] = 0

dupe_check = combined_mesh.n_points == (cartilage_remesh.n_points+bone_mesh.n_points) - mesh_edge_ids.shape[0]
if not dupe_check:
    raise AssertionError('Not all duplicate points removed')
################# GET FINAL COMBINED MESH WITH REGION IDs #################

################# MESH CHECKS #################
# get cells ids of each region on the combined mesh
combined_mesh_cell_ids = np.arange(combined_mesh.n_cells)
bone_surf_ids = combined_mesh_cell_ids[np.where(combined_mesh['region_id']==1)[0]]
bone_shell_ids = combined_mesh_cell_ids[np.where(combined_mesh['region_id']!=2)[0]]
cartilage_surf_ids = combined_mesh_cell_ids[np.where(combined_mesh['region_id']==2)[0]]
#interface_surf_ids = combined_mesh_cell_ids[np.where(combined_mesh['region_id']==3)[0]]

# extract enclosed cartilage volume to do checks
cartilage_mesh = combined_mesh.extract_cells(bone_surf_ids, invert=True).extract_surface(algorithm=None)
remove_normals(cartilage_mesh)  # have to remove them before recomputing 
                        #- cos inherited from parent mesh and doesn't recompute something about them for some reason
cartilage_mesh.compute_normals(auto_orient_normals=True, inplace=True)
# check if normals point outwards or inwards and if not flip them
if not get_outward_normal_mask(cartilage_mesh.cell_centers().points, cartilage_mesh.cell_normals, cartilage_mesh).any():
    cartilage_mesh = flip_faces(cartilage_mesh, np.arange(cartilage_mesh.n_cells))
    print('flipped faces')

print('\nMESH CHECKS...')
# mesh checks
print('\n----- CARTILAGE -----')
mesh_checks(cartilage_mesh, raise_error=True)

print('\n----- BONE -----')
mesh_checks(bone_mesh, raise_error=True)

print('\n----- BONE-CARTILAGE -----')
bone_cartilage_checks(bone_mesh, cartilage_mesh, interface_mesh.points, raise_error=True, check_intersection=False)

combined_mesh_centres = combined_mesh.cell_centers().points
combined_mesh_normals = combined_mesh.cell_normals

# check if normals point outwards
bone_normal_mask = get_outward_normal_mask( # including boundary
    combined_mesh_centres[bone_shell_ids], 
    combined_mesh_normals[bone_shell_ids], 
    bone_mesh
    )
cartilage_normal_mask = get_outward_normal_mask(
    combined_mesh_centres[cartilage_surf_ids], 
    combined_mesh_normals[cartilage_surf_ids], 
    cartilage_mesh
    )

# check if all normals point outwards, fix if not
flip_ids = np.hstack((cartilage_surf_ids[~cartilage_normal_mask], bone_shell_ids[~bone_normal_mask]))
if len(flip_ids):
    combined_mesh_flipped = flip_faces(combined_mesh, flip_ids)
    # changed flip faces to carry over point/cell arrays 2026-03-11
else:
    combined_mesh_flipped = combined_mesh

# check normal direction
if not get_outward_normal_mask( 
    combined_mesh_centres[bone_shell_ids], 
    combined_mesh_flipped.cell_normals[bone_shell_ids], 
    bone_mesh
    ).all():
    raise AssertionError('Not all bone normals point outward')
if not get_outward_normal_mask( 
    combined_mesh_centres[cartilage_surf_ids], 
    combined_mesh_flipped.cell_normals[cartilage_surf_ids], 
    cartilage_mesh
    ).all():
    raise AssertionError('Not all cartilage surface normals point outward')

# manually check normals
faces = combined_mesh_flipped.faces.reshape(-1, 4)[:, 1:]
points = combined_mesh_flipped.points

v1 = points[faces[:, 1]] - points[faces[:, 0]]
v2 = points[faces[:, 2]] - points[faces[:, 0]]
geom_normals = np.cross(v1, v2)
geom_normals /= np.linalg.norm(geom_normals, axis=1)[:, None]

dots = np.einsum("ij,ij->i", geom_normals, combined_mesh_flipped['Normals'])
if not np.min(dots)>0.999:
    raise AssertionError("['Normals'] don't match edge winding")
# mesh['Normals']doesn't update with edge winding but .cell_normals does
################# MESH CHECKS #################

################# RETURN MESH #################
#return mesh_clean, cartilage_cap, combined_mesh_flipped #  before smoothing, before remeshing, final bone cartilage shared interface
################# RETURN MESH #################

Created cartilage cap
Smoothing cartilage cap
Writing mesh to remeshing input directory
Remeshing cartilage cap
Remeshing complete.

Attaching cartilage cap

MESH CHECKS...

----- CARTILAGE -----
Trimesh checks:
Mesh is watertight               True
Mesh is winding consistent       True

PyVista checks:
Mesh is manifold (no open edges) True
All normals point outwards:      True
No duplicate faces               True
No duplicate points              True

----- BONE -----
Trimesh checks:
Mesh is watertight               True
Mesh is winding consistent       True

PyVista checks:
Mesh is manifold (no open edges) True
All normals point outwards:      True
No duplicate faces               True
No duplicate points              True

----- BONE-CARTILAGE -----
All interface nodes present and identical     True


In [14]:
pv.plot(combined_mesh)

Widget(value='<iframe src="http://localhost:51240/index.html?ui=P_0x1764dc110_0&reconnect=auto" class="pyvista…

In [21]:
pl = pv.Plotter()
pl.add_points(midpoints, color='black', point_size=15)
pl.add_mesh(bone_mesh)
pl.show()

Widget(value='<iframe src="http://localhost:51240/index.html?ui=P_0x34ae922a0_6&reconnect=auto" class="pyvista…

#### mesh the hight field

In [60]:
midpoint_mesh = pv.PolyData(midpoints, faces=inter_mesh.faces)
midpoint_mesh['mesh_id'] = -min_df['bone_id'].values
inter_mesh['mesh_id'] = inter_mesh['bone_id']

from phd_helpers.CartilageGeneration import stitch_cartilage
bone_surf = bone_mesh.extract_cells(inter_mesh['bone_cell_id'], invert=True)
stitched_cart = stitch_cartilage(midpoint_mesh, inter_mesh)
stitched_mesh = stitched_cart + bone_surf
stitched_mesh['region_id'] = np.full(stitched_mesh.n_cells, 2)
stitched_mesh['region_id'][find_corresponding_cells(stitched_mesh, inter_mesh)] = 3
stitched_mesh['region_id'][find_corresponding_cells(stitched_mesh, bone_surf)] = 1

In [61]:
stitched_mesh.plot()

Widget(value='<iframe src="http://localhost:51240/index.html?ui=P_0x355240b00_15&reconnect=auto" class="pyvist…

In [54]:
def get_rotation_matrix(vec, angle):
    """
    Define a local coordinate system with:
        z-axis = vec
        x-axis = angle rotation about z from projected global x-axis
        y-axis = cross(z, x)
        Rotation matrix - columns: [x_axis, y_axis, z_axis].
    """
    import numpy as np

    z_axis = np.asarray(vec, dtype=float)
    z_axis /= np.linalg.norm(z_axis)

    # Choose a global reference axis not parallel to z
    ref = np.array([1.0, 0.0, 0.0])
    if abs(np.dot(ref, z_axis)) > 0.99:
        ref = np.array([0.0, 1.0, 0.0])

    # Project reference onto plane perpendicular to z
    x0 = ref - np.dot(ref, z_axis) * z_axis
    x0 /= np.linalg.norm(x0)

    y0 = np.cross(z_axis, x0)
    y0 /= np.linalg.norm(y0)

    # Rotate x about z
    theta = np.deg2rad(angle)
    x_axis = np.cos(theta) * x0 + np.sin(theta) * y0
    x_axis /= np.linalg.norm(x_axis)

    # Complete right-handed basis
    y_axis = np.cross(z_axis, x_axis)
    y_axis /= np.linalg.norm(y_axis)

    return np.column_stack((x_axis, y_axis, z_axis))

In [105]:
from phd_helpers.paths import transform_mesh, transform_points

angle = 10
centre = inter_mesh.points.mean(axis=0)
vec = inter_mesh.cell_normals[inter_mesh.find_closest_cell(centre)]
R = get_rotation_matrix(vec, angle)

mesh1 = transform_mesh(combined_mesh.extract_cells(combined_mesh['region_id']==2, invert=True), R, centre, inverse=True)
points1 = transform_points(midpoints, R, centre, inverse=True)
mesh2 = transform_mesh(stitched_mesh, R, centre, inverse=True)
mesh3 = transform_mesh(combined_mesh, R, centre, inverse=True)


In [106]:
def slice_mesh(mesh, v=0):
    mask = mesh.cell_centers().points[:, 1] < v
    return mesh.extract_cells(mask)

def slice_points(points, v=0):
    mask = points[:, 1] < v
    return points[mask]

In [132]:
camera_position = pv.CameraPosition(position=(-6.3909592987958375, -15.613164264459744, 1.1047493364747996),
               focal_point=(-4.122223176980405, -4.572263601866421, 0.4325207168946802),
               viewup=(0.15798442182452013, 0.027633756636214618, 0.9870548606612211))

ps = 15

from matplotlib.colors import ListedColormap

colors = [
    "#484744",  # region 1
    "#C28A3B",  # region 2
    "#3B874D",  # region 3
]
cmap = ListedColormap(colors)

In [141]:
pl = pv.Plotter()
pl.enable_anti_aliasing('ssaa')
pl.add_points(slice_points(points1), color='black', point_size=ps)
pl.add_mesh(slice_mesh(mesh1), scalars='region_id', cmap=cmap, show_scalar_bar=False)
pl.camera_position = camera_position
pl.show()

Widget(value='<iframe src="http://localhost:51240/index.html?ui=P_0x3c42512b0_67&reconnect=auto" class="pyvist…

In [142]:
pl.screenshot('figs/mesh1.png', scale=2, transparent_background=False, return_img=False)

In [139]:
pl = pv.Plotter()
pl.enable_anti_aliasing('ssaa')
pl.add_points(slice_points(points1), color='black', point_size=ps)
pl.add_mesh(slice_mesh(mesh2, 0), scalars='region_id', cmap=cmap, show_scalar_bar=False)
pl.camera_position = camera_position
pl.show()

Widget(value='<iframe src="http://localhost:51240/index.html?ui=P_0x3abf90500_66&reconnect=auto" class="pyvist…

In [140]:
pl.screenshot('figs/mesh2.png', scale=2, transparent_background=False, return_img=False)

In [138]:
pl = pv.Plotter()
pl.enable_anti_aliasing('ssaa')
pl.add_points(slice_points(points1), color='black', point_size=ps)
pl.add_mesh(slice_mesh(mesh3), scalars='region_id', cmap=cmap, show_scalar_bar=False)
pl.camera_position = camera_position
pl.show()

Widget(value='<iframe src="http://localhost:51240/index.html?ui=P_0x3af63e420_65&reconnect=auto" class="pyvist…

In [134]:
pl.screenshot('figs/mesh3.png', scale=2, transparent_background=False, return_img=False)

In [ ]:
tranform into inertial axis
slice the geometry 
take 3 screenshots
 - have cartilage region, determined by 2 mm proximity, highlighted
 - for sharp ridge image mention that ridge can be normal to surface forming 90 angle, or more or less.
    - can also interfere
    - and just attaching to nearest point, is not adhereing to 2mm proximity